# Breast cancer tumor classification

**Executive summary:**

This notebook presents a classification model to predict whether a breast tumor is malignant or benign using the Wisconsin Diagnostic Breast Cancer (WDBC) dataset. I began with structured feature engineering informed by domain knowledge of tumor morphology, creating new aggregated and interaction features based on cell radius, area, concavity, and related metrics. After robust preprocessing, scaling, and removal of highly correlated features, I trained and evaluated several models, ultimately selecting XGBoost for its ability to capture non-linear relationships.

The model achieved a cross-validated F1 score of 0.98 and high accuracy (97%) on the test set, demonstrating strong generalization and effective use of biologically meaningful patterns. Classification metrics indicate the model performs well without overfitting.

More details on preprocessing, feature engineering, modeling decisions, and evaluation can be found in the sections below.

**Dataset details:**

Use the dataset found at https://archive.ics.uci.edu/dataset/17/breast+cancer+wisconsin+diagnostic to classify tumors as benign or malignant.

Features are computed from a digitized image of a fine needle aspirate (FNA) of a breast mass.  They describe characteristics of the cell nuclei present in the image. It's important to note that each of the 10 feature types is measured in 3 ways:
* _mean – the average value across the sample
* _se – the standard error (i.e., variability)
* _worst – the largest value (mean of top 3 largest cell values)

They are different aspects of the same thing, so we should watch out for multicollinearity.

**Assumptions/next steps:**
* **General code setup: everything is in notebooks right now**
    * Helper functions are currently located at the top of the notebook
    * With more time, they'd be pulled into source code `.py` files, with separate classes for data loading/preprocessing and modeling
        * If you want to assess my ability to write good source code, please visit my GitHub at https://github.com/khiller17
    * I'd use sklearn pipelines where possible instead of doing it all step-by-step
    * Justification:
        * Wanted to devote most time to EDA/feature engineering/model tuning
        * Wanted to visualize the data at multiple steps in the notebook
        * This isn't in prod
* See markdown in subsequent sections for a couple of additional next steps

In [ ]:
import re
import pandas as pd
import numpy as np
import plotly.express as px
import seaborn as sns
from scipy import stats
import matplotlib.pyplot as plt

from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

from src.load import load_data
from src.preprocess import add_aggregated_features, add_interaction_features, drop_correlated_features
from src.utils import pointbiserial_correlation, plot_confusion_matrix

## Clean and explore the dataset

Observations:
* Features are on different scales - scaling the data will be critical here
* No missing values
* Class imbalance in favor of benign samples - consider weighting, over/undersampling, SMOTE
* Some features have outliers that might be biologically significant - consider robust scaler

In [ ]:
df = load_data("./data/wdbc.data")
df.shape

In [ ]:
df.head()

In [ ]:
df.describe()

In [ ]:
df.diagnosis.value_counts()

## Feature engineering/selection

### Add engineered features

The dataset is clean and compact and so it might not require much feature engineering. That said, it's worth trying the following:

* Create interaction features (might better reflect cell shape or morphology) - these are done just based on intuition from common knowledge (malignant tumors tend to have larger, more irregular, and more variable nuclei) that I got from Google/literature, and I used interaction effects a lot in grad school
    * area_mean / perimeter_mean (a proxy for compactness/circularity - malignant cells likely to be oddly shaped??)
    * concavity_mean * compactness_mean (joint measure of irregularity - highlights cases where cells have a jagged edge and are indented, both signs of malignancy)
    * concave_points_worst / concave_points_mean (change from mean to worst - cancer cells are heterogenous!)
        * how much the most distorted cells deviate from the average distortion in the same sample
        * high ratio means a few extremely abnormal cells exist in the sample, else it's more even
* Feature aggregation - reduce multicollinearity and emphasize biologically important things like how much a feature changes between average and worst (normal and extreme)
    * These are simple aggregation features that summarize patterns across the different variables measured
    * df["radius_range"] = df["radius_worst"] - df["radius_mean"] (capture tumor heterogeneity)
    * df["texture_variability"] = df["texture_se"] / df["texture_mean"] (capture inconsistency in these values)

In [ ]:
df = add_aggregated_features(df)
df.shape

In [ ]:
df = add_interaction_features(df)
df.shape

### Visualize the data

You can get an idea of data distributions if you want to do any statistical analyses here. You can also see if there are visual differences between diagnoses, which might tell you which features will be important to include in the model. You can also spot outliers, which can inform how you scale the data and also whether you should consider outlier removal.

Note that we just show a subset of features here to save space. You can view others by changing the values in the cols_to_plot list. With more time I'd put this into a function instead of hard-coding.

Observations:
* Some features like radius_mean are clearly different between diagnoses
* Others like texture_mean are less different between diagnoses
* Sometimes when there is a difference between diagnoses, the malignant samples show higher standard deviation (indicating more heterogeneity in cancer cells) in their value distributions than benign samples
    * see area_mean, radius_mean etc
* Sometimes there is no difference in standard devation between diagnoses (e.g. texture_mean)

In [ ]:
cols_to_plot = ["area_mean", "concavity_mean", "concave_points_worst", "radius_mean", "perimeter_mean", "texture_mean"]
for col in df[cols_to_plot].columns:
    sns.histplot(data=df, x=col, hue="diagnosis", kde=True, element="step", stat="density")
    plt.title(f"{col} distribution by diagnosis")
    plt.show()

### Perform train/test split

Split into train and test sets. We do this to reserve a holdout set for model evaluation. This must not be involved in model selection at all, including feature selection and model training/cross validation in order to avoid leakage. This gives us an idea of how well the model will generalize to unseen data.

We do not want to do things like scaling or selecting features on samples from the holdout set. This needs to be done early.

Since we have imbalanced classes, it makes sense to stratify the split to maintain the original class distribution.

In [ ]:
X = df.drop(['id', 'diagnosis'], axis=1)
y = df[['diagnosis']]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=1738, stratify=y)
X_train.shape, X_test.shape, y_train.shape, y_test.shape

### Scale

Features are on different scales, and some models (e.g. SVM) are not scale-invariant. In that case, features that are on higher scales might get more weight in the model just becasue their values are bigger, and not because they're actually more discriminative between the target classes. For example, a feature that ranges from 100-200 might have more effect in an SVM than a feature that ranges from 0-2. Scaling addresses this issue.

In [ ]:
scaler = RobustScaler()
X_train_tf = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns.values.tolist(), index=X_train.index)
X_test_tf = pd.DataFrame(scaler.transform(X_test), columns=X_test.columns.values.tolist(), index=X_test.index)
X_train_tf.shape, X_test_tf.shape

### Visualize correlations and remove highly correlated features

Here we see if any features are highly correlated. This is defined as having an absolute correlation value of >0.9. Removing correlated features can reduce overfitting in models like SVMs. This is because these features are so similar that it's almost like including the same feature multiple times, which lets it influence the model more than it should. This is less of an issue in other models like random forests because trees are built independently with different feature subsets, but since we want to compare SVM with random forest and XGBoost, and decorrelating doens't hurt in ensemble methods, we can go ahead and do it.

As expected, things like radius mean and radius worst are highly correlated. 

Interestingly, some things like radius_avg (avg nucleus size - cancer cells have bigger nuclei) and concavity_rel_var (depth of indentations relative to the avg) are negatively correlated. This might be because larger nuclei tend to exhibit consistently irregular contours, leading to lower relative variability in concavity, while smaller nuclei often show more erratic concavity relative to their size, resulting in a negative correlation between radius_avg and concavity_rel_var.

In [ ]:
corrmat = X_train_tf.corr()
fig = px.imshow(corrmat, 
                labels=dict(x="Features", y="Features", color="Correlation"),
                x=corrmat.columns,
                y=corrmat.index,
                color_continuous_scale=px.colors.diverging.RdBu,
                range_color=[-1, 1])
fig.update_layout(title_text='Correlation Matrix', width=1000, height=1000)
fig.show()

In [ ]:
# now drop highly correlated features
to_drop = drop_correlated_features(corrmat)
X_train_tf = X_train_tf.drop(to_drop, axis=1)
X_test_tf = X_test_tf.drop(to_drop, axis=1)
X_train_tf.shape, X_test_tf.shape

### Examine feature discrimination

Which features separate benign from mutation data well? Can measure this with point biserial correlation. This is appropriate when comparing a binary and a continuous variable.

In [ ]:
correlations = X_train_tf.corrwith(y_train.diagnosis, method=pointbiserial_correlation)
correlations = correlations.sort_values(ascending=False)
correlations = pd.DataFrame(correlations, columns=['correlation']).reset_index().rename(columns={'index':'feature'})

In [ ]:
fig = px.bar(
    correlations,
    x="correlation",
    y="feature",
    orientation="h",
    title="Feature–Target Correlation (Diagnosis = 1 = Malignant)",
    labels={"correlation": "Correlation with Malignancy", "feature": "Feature"},
    color="correlation",
    color_continuous_scale="RdBu",
    range_color=[-1, 1],
    width=800,
    height=700
)

fig.update_layout(yaxis=dict(autorange="reversed"))
fig.show()

### Feature selection

This isn't a high-dimensional dataset, so I don't think we have to worry much about doing any fancy feature selection, especially with models that are robust to high-dim data like SVM or random forest. But we can (and should) remove features with very little to no correlation with our target variable since those are basically just noise. Let's remove features with absolute values of correlations below 0.05.

In [ ]:
correlations = correlations[correlations['correlation'].abs() >= 0.05]
correlations.shape

In [ ]:
X_train_tf = X_train_tf[correlations.feature.values.tolist()]
X_test_tf = X_test_tf[correlations.feature.values.tolist()]
X_train_tf.shape, X_test_tf.shape

## Train model

I compared SVM with random forest and XGBoost. XGBoost had the best performance, so I am keeping those results here. I chose XGBoost because it can handle any nonlinear relationships in the dataset, and because it boosted accuracy relative to random forest, likely to to its sequential tree-building method (it corrects errors in trees that came before it). Since we are not overfitting, I think it's safe and accurate to use it on even this small dataset. Plus, it has built-in feature importances that we can use to explain how the model made decisions to regulatory bodies if necessary. It's also SHAP/LIME-friendly for explainability purposes as well.

I fit my xgboost model while emphasizing the positive class due to imbalance using the `scale_pos_weight` parameter. I performed GridSearchCV to perform 5 fold CV and select best hyperparameters for my model.

Results:
* Holdout accuracy is 97% which puts us in a good range based on results on the dataset webpage
* This level of holdout accuracy also means we are not overfitting, especially given good balance between precision and recall we are seeing in the classification report
* We lose a little precision/recall in the positive class likely due to class imbalance
    * Class weighing can only fix so much - with more time some additional methods to balance classes like oversampling or SMOTE might make sense

### Train

In [ ]:
# calculate scale pos weight for xgboost - makes model pay more attention to malignant cases
neg, pos = np.bincount(y_train["diagnosis"])
scale_pos_weight = neg / pos
print(f"scale_pos_weight: {scale_pos_weight:.2f}")

In [ ]:
param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.1, 0.2],
    'subsample': [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0],
    'gamma': [0, 0.1, 0.5],
    'reg_lambda': [1, 10],  # L2 regularization
    'reg_alpha': [0, 1],    # L1 regularization
}
xgb = XGBClassifier(
    objective='binary:logistic',
    use_label_encoder=False,
    eval_metric='logloss',  # suppress warning
    random_state=1738,
    scale_pos_weight=scale_pos_weight,
    verbosity=0
)

grid_search_xgb = GridSearchCV(
    estimator=xgb,
    param_grid=param_grid,
    scoring='f1',
    cv=5,
    n_jobs=-1,
    verbose=2
)

grid_search_xgb.fit(X_train_tf, y_train["diagnosis"])

# Best score + params
print("Best parameters:", grid_search_xgb.best_params_)
print("Best F1 score (CV):", grid_search_xgb.best_score_)

### Assess performance

In [ ]:
y_pred_xgb = grid_search_xgb.best_estimator_.predict(X_test_tf)
print(classification_report(y_test["diagnosis"], y_pred_xgb))

In [ ]:
y_pred = grid_search_xgb.best_estimator_.predict(X_test_tf)
plot_confusion_matrix(y_test["diagnosis"], y_pred, title="XGBoost Confusion Matrix")

### Feature importance

Which features contributed most to model decision making? Do they make biological sense?

Most important feature is radius_range, which tells us two things:
1. My engineered features were helpful (or at least some of them!)
2. This is the differnce between radius mean and radius worst, so it's a good measure of size heterogeneity
    * Malignant tumors usually have more heterogeniety in cell size
  
Second most important is concavity_mean, which also makes biological sense because we know that cancer cells have irregular, rough shapes and outlines.

These feature importances can help us sanity check the model, explain how it's making decisions, and give us insight into what key identifiers mark malignant tumors. For exmaple, we learned that cell heterogeneity is important through an engineered feature.

Given more time, a more sophisticated method like LIME or permutation tests would be in order to gain a more robust understanding of explainability.

In [ ]:
best_xgb = grid_search_xgb.best_estimator_
importances = pd.Series(best_xgb.feature_importances_, index=X_train_tf.columns)
top_n = 15  
top_features = importances.sort_values(ascending=False).head(top_n)

plt.figure(figsize=(10, 6))
top_features.plot(kind='barh')
plt.gca().invert_yaxis()
plt.title("Top XGBoost Feature Importances")
plt.xlabel("Importance Score")
plt.tight_layout()
plt.show()

## Summary/conclusions

This pipeline, which combined robust preprocessing, biologically-informed feature engineering, and class-aware hyperparameter tuning, resulted in a highly performant XGBoost model capable of distinguishing between benign and malignant breast tumors with near state-of-the-art accuracy (CV F1 = 0.981). We achieved 97% accuracy overall on the holdout set, with 0.95 and 0.97 F1 scores for malignant and benign classes respectively. This indicates not only good performance but also that the model shows no signs of overfitting. Our recall on holdout for malignant cases is 0.94 which in my opinion is good for imbalanced classes, and shows the model is cautious about missing malignant cases. I tuned it to increase this number because missing malignant cases is likely the most dangerous scenario for patients. That said, the precision value for malignant of 0.96 also shows that we are rarely labeling benign cases as malignant and giving people false cancer scares.

This analysis demonstrates how thoughtful feature construction and class imbalance handling can significantly enhance performance in medical classification tasks.